<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop/blob/main/Agentic_Movie_Storyboarding_Living_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Agentic Movie Storyboarding — Neo4j + Graphiti Living Agents

A **minimal, agentic, human-in-the-loop, self-reinforcing** workflow for movie storyboarding built on top of an existing **Neo4j movie graph**, powered by:
- **OpenAI** (chat + embeddings) for every creative agent,
- **`neo4j-graphrag`** `HybridCypherRetriever` (vector + fulltext fusion + `HAS_GENRE` Cypher expansion) for the Movie Intelligence Agent,
- **🌐 Graphiti** as the **single memory backbone** — every artifact, every piece of director feedback, every approval is logged as a temporal *episode*. Graphiti's LLM auto-extracts `:Entity` nodes and `:RELATES_TO` edges with bi-temporal `valid_at`/`invalid_at`, all stored in the **same Neo4j database** as `:Movie`/`:Genre`.
- **Mandatory free-text feedback before every agent acts** — captured, classified, and logged to Graphiti. Graphiti automatically extracts director-preference facts; newer feedback that contradicts older facts gets the older facts marked `invalid_at` (no manual `:SUPERSEDES`).
- 🔁 **Cross-session living loop** — at the start of every session, `graphiti.search(...)` recalls the director's most relevant past facts; those are folded into the Movie Intelligence query text **and** the Creative Direction Agent's prompt, so the **same** hybrid retriever immediately surfaces movies aligned with the director's accumulated taste.

**Roles**
1. 🧑‍🎨 **Human Creative Director** *(you)* — supplies the wild idea **and gives free-text feedback at every checkpoint**.
2. 🧠 **Movie Intelligence Agent** — hybrid GraphRAG search, query text steered by Graphiti recall.
3. 🎨 **Creative Direction Agent** — LLM proposes 3 directions, grounded in retrieved movies + Graphiti facts about the director.
4. ✍️ **Scripting Agent** — LLM drafts a 3-act outline; re-run if disliked.
5. 🎞️ **Screenplay Agent** — LLM converts the script to scenes; re-run if disliked.
6. 🖼️ **Storyboarding Agent** — LLM turns the screenplay into panels; re-run if disliked.
7. 🌐 **Memory Layer** — **Graphiti** logs every artifact + feedback + approval as a temporal episode; bi-temporal facts capture how the director's taste evolves.

User input is collected via `input()` and `getpass.getpass()`.


## 1. Setup
Install the Neo4j Python driver if needed and import dependencies.

In [ ]:
# Run once if not already installed
%pip install neo4j "neo4j-graphrag[openai]" openai pandas python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 19.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import uuid
import getpass
from datetime import datetime

import pandas as pd
from neo4j import GraphDatabase

# neo4j-graphrag: hybrid retrieval (vector + fulltext) + LLM + embedder
from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.indexes import (
    create_vector_index,
    create_fulltext_index,
    upsert_vectors,
)
from neo4j_graphrag.types import EntityType
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

## 2. Connect to Neo4j + OpenAI
Credentials are collected securely with `getpass`.
- `NEO4J_URI` → your movie-graph instance (e.g., `neo4j+s://<dbid>.databases.neo4j.io` for Aura, or `bolt://localhost:7687` locally).
- **OpenAI** is used for the LLM (chat) and the embedder (text-embedding) that powers **hybrid search**.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/.env')
except Exception:
    print("error reading env details")
    pass

# --- Neo4j ---
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

# --- OpenAI ---
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")
OPENAI_CHAT_MODEL  = "gpt-5-mini"
OPENAI_EMBED_MODEL = "text-embedding-3-small"
EMBEDDING_DIM      = 1536

# Make the key available to the OpenAI SDK (used internally by neo4j-graphrag)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j at", NEO4J_URI)

✅ Connected to Neo4j at bolt://34.201.26.185


In [ ]:
# Quick smoke test against the existing movie graph
with driver.session(database=NEO4J_DATABASE) as session:
    df = session.run(
        """
        MATCH (m:Movie)
        RETURN count(m) AS movies
        """
    ).to_df()
df

,movies
0,496


## 2b. 🤖 Initialise the LLM and Embedder
We use **OpenAI** for both the chat model (driving every agent) and the embedding model (powering vector search). These are wrapped by `neo4j-graphrag` so the retrievers can call them directly.

In [ ]:
# --------------------------------------------------------------------------- #
# 🧠 Detect "reasoning" models (gpt-5*, o1-*, o3-*, o4-*).                    #
# These reject the legacy `max_tokens` and REQUIRE `max_completion_tokens`.  #
# They also don't accept `temperature` — sampling is fixed internally.        #
# --------------------------------------------------------------------------- #
_is_reasoning_model = any(
    p in OPENAI_CHAT_MODEL.lower() for p in ("gpt-5", "o1-", "o3-", "o4-")
)
if _is_reasoning_model:
    _chat_params = {"max_completion_tokens": 2000}
    print(f"🧠 Detected reasoning model {OPENAI_CHAT_MODEL!r} — using max_completion_tokens.")
else:
    _chat_params = {"temperature": 0.7, "max_tokens": 1500}

🧠 Detected reasoning model 'gpt-5-mini' — using max_completion_tokens.


In [ ]:
# Chat model — powers all creative agents
llm = OpenAILLM(
    model_name=OPENAI_CHAT_MODEL,
    api_key=OPENAI_API_KEY,
    model_params=_chat_params,
)

# Embedder — used to embed movies AND user queries for vector/hybrid search
embedder = OpenAIEmbeddings(
    model=OPENAI_EMBED_MODEL,
    api_key=OPENAI_API_KEY,
)

In [ ]:
# 🛡️ Safety net: if neo4j-graphrag (or any other library) tries to pass
# `max_tokens` to a reasoning model, transparently rewrite it to
# `max_completion_tokens` at the OpenAI client layer. This makes the LLM
# robust to version drift in upstream libraries.
if _is_reasoning_model:
    import functools as _ft
    def _patch_chat_create(client):
        _orig = client.chat.completions.create
        @_ft.wraps(_orig)
        def _wrapped(*args, **kwargs):
            if "max_tokens" in kwargs and "max_completion_tokens" not in kwargs:
                kwargs["max_completion_tokens"] = kwargs.pop("max_tokens")
            else:
                kwargs.pop("max_tokens", None)
            # reasoning models also reject custom temperature
            if kwargs.get("temperature") not in (None, 1, 1.0):
                kwargs.pop("temperature", None)
            return _orig(*args, **kwargs)
        client.chat.completions.create = _wrapped

    for _client_attr in ("client", "async_client"):
        _c = getattr(llm, _client_attr, None)
        if _c is not None:
            _patch_chat_create(_c)
    print("🛡️ Patched OpenAI client to translate max_tokens → max_completion_tokens.")

🛡️ Patched OpenAI client to translate max_tokens → max_completion_tokens.


In [ ]:
# Sanity check
_test_vec = embedder.embed_query("a test sentence")
print(f"✅ Embedder ready — vector dim = {len(_test_vec)} (expected {EMBEDDING_DIM})")
print(f"✅ LLM ready — model = {OPENAI_CHAT_MODEL}")

✅ Embedder ready — vector dim = 1536 (expected 1536)
✅ LLM ready — model = gpt-5-mini


In [ ]:
# --------------------------------------------------------------------------- #
# 🎨 Display utilities — for a hack-session-ready visual showcase             #
# --------------------------------------------------------------------------- #
# Replaces walls of `print(...)` with color-coded HTML cards, collapsible
# artifact blocks, sentiment pills, and stage banners. Also silences the
# noisy INFO logs from httpx / openai / graphiti that would otherwise drown
# the demo output.
# --------------------------------------------------------------------------- #
import logging
from html import escape as _esc
from IPython.display import HTML, display

# 🔇 Silence noisy library logs (httpx HTTP requests, openai retries,
# graphiti notifications) so the demo output stays clean.
for _ln in ("httpx", "httpcore", "openai", "neo4j", "neo4j.notifications",
            "graphiti_core", "graphiti_core.llm_client",
            "graphiti_core.llm_client.openai_base_client"):
    logging.getLogger(_ln).setLevel(logging.WARNING)

# 🎨 Catppuccin-Mocha-inspired palette: one (foreground, background) per role.
_PALETTE = {
    "intelligence": ("#89b4fa", "#1e2030"),  # blue   — Movie Intelligence
    "direction":    ("#f5c2e7", "#2a1e2a"),  # pink   — Creative Direction
    "script":       ("#a6e3a1", "#1e2a1e"),  # green  — Scripting
    "screenplay":   ("#fab387", "#2a2218"),  # orange — Screenplay
    "storyboard":   ("#cba6f7", "#251e2a"),  # purple — Storyboarding
    "memory":       ("#94e2d5", "#1e2a26"),  # teal   — Graphiti memory
    "human":        ("#f9e2af", "#2a2718"),  # yellow — Human checkpoint
    "approve":      ("#a6e3a1", "#1e2a1e"),  # green  — Approved
    "revise":       ("#f38ba8", "#2a1e22"),  # red    — Revision needed
}


def session_banner(session_id: str) -> None:
    """Big banner shown at the very start of a workflow run."""
    fg, _ = _PALETTE["intelligence"]
    display(HTML(f"""
    <div style="margin:18px 0;padding:22px 26px;
                background:linear-gradient(135deg,#1e1e2e 0%,#11111b 100%);
                border:2px solid {fg}88;border-radius:14px;text-align:center;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:30px;">🎬</div>
      <div style="font-size:24px;color:#cdd6f4;font-weight:700;margin-top:6px;
                  letter-spacing:0.5px;">Agentic Movie Storyboarding</div>
      <div style="font-size:12px;color:{fg};margin-top:8px;letter-spacing:2px;
                  text-transform:uppercase;opacity:0.85;font-weight:600;">
        Session · {_esc(session_id)}
      </div>
    </div>
    """))


def stage_banner(stage_num: int, title: str,
                 color_key: str = "intelligence", subtitle: str = "") -> None:
    """Big gradient banner that marks the start of a workflow stage."""
    fg, bg = _PALETTE[color_key]
    sub_html = (f'<div style="font-size:13px;color:#9399b2;margin-top:6px;">{_esc(subtitle)}</div>'
                if subtitle else '')
    display(HTML(f"""
    <div style="margin:22px 0 12px 0;padding:18px 22px;
                background:linear-gradient(135deg,{bg} 0%,#181825 100%);
                border-left:5px solid {fg};border-radius:10px;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:11px;letter-spacing:2.5px;color:{fg};opacity:0.75;
                  text-transform:uppercase;font-weight:700;">STAGE {stage_num}</div>
      <div style="font-size:22px;color:#cdd6f4;font-weight:600;margin-top:4px;">{title}</div>
      {sub_html}
    </div>
    """))


def artifact_card(title: str, body: str, color_key: str = "script",
                  icon: str = "📄", collapsed: bool = False) -> None:
    """Collapsible artifact card — Script / Screenplay / Storyboard output."""
    fg, bg = _PALETTE[color_key]
    open_attr = "" if collapsed else " open"
    display(HTML(f"""
    <details{open_attr} style="margin:10px 0;border:1px solid {fg}55;
              border-radius:10px;background:{bg};overflow:hidden;
              font-family:'Segoe UI',sans-serif;">
      <summary style="cursor:pointer;padding:12px 18px;
                background:linear-gradient(90deg,{bg} 0%,#181825 100%);
                color:{fg};font-weight:600;font-size:14px;
                display:flex;align-items:center;gap:10px;list-style:none;">
        <span style="font-size:20px;">{icon}</span>
        <span>{_esc(title)}</span>
        <span style="margin-left:auto;font-size:11px;opacity:0.6;font-weight:400;">
          click to expand / collapse
        </span>
      </summary>
      <div style="padding:16px 20px;background:#11111b;color:#cdd6f4;
                  font-family:'Cascadia Code','Consolas',monospace;font-size:13px;
                  line-height:1.65;white-space:pre-wrap;">{_esc(body)}</div>
    </details>
    """))


def checkpoint_banner(stage: str, preview: str | None = None) -> None:
    """Human-in-the-loop checkpoint header — replaces the dashed print banner."""
    fg, bg = _PALETTE["human"]
    safe_preview = ""
    if preview:
        snippet = preview if len(preview) <= 600 else preview[:600] + "…"
        safe_preview = f"""
        <div style="margin-top:12px;padding:10px 14px;background:#11111b;
                    border-radius:6px;border:1px solid {fg}33;
                    font-family:'Cascadia Code','Consolas',monospace;
                    font-size:12px;color:#9399b2;line-height:1.55;
                    white-space:pre-wrap;max-height:220px;overflow:auto;">{_esc(snippet)}</div>"""
    display(HTML(f"""
    <div style="margin:16px 0;padding:16px 20px;
                background:linear-gradient(135deg,{bg} 0%,#181825 100%);
                border:1px solid {fg}66;border-radius:10px;
                font-family:'Segoe UI',sans-serif;">
      <div style="display:flex;align-items:center;gap:12px;">
        <span style="font-size:24px;">🧑‍🎨</span>
        <div>
          <div style="font-size:11px;letter-spacing:1.8px;color:{fg};
                      text-transform:uppercase;font-weight:700;opacity:0.85;">
            Human Checkpoint
          </div>
          <div style="font-size:16px;color:#cdd6f4;font-weight:600;">{_esc(stage)}</div>
        </div>
      </div>{safe_preview}
    </div>
    """))


def feedback_pill(text: str, sentiment: str,
                  change_requested: bool, summary: str) -> None:
    """Captured-feedback card with colored sentiment + action pills."""
    sentiment = (sentiment or "neutral").lower()
    if sentiment == "positive":
        s_fg, s_bg = _PALETTE["approve"]
    elif sentiment == "negative":
        s_fg, s_bg = _PALETTE["revise"]
    else:
        s_fg, s_bg = "#bac2de", "#313244"
    c_fg, c_bg = _PALETTE["revise"] if change_requested else _PALETTE["approve"]
    c_label = "REVISE" if change_requested else "APPROVE"
    display(HTML(f"""
    <div style="margin:8px 0 16px 0;padding:14px 18px;background:#181825;
                border-radius:8px;border-left:3px solid {s_fg};
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:13px;color:#cdd6f4;margin-bottom:10px;
                  font-style:italic;line-height:1.55;">&ldquo;{_esc(text)}&rdquo;</div>
      <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
        <span style="background:{s_bg};color:{s_fg};padding:3px 11px;
                     border-radius:12px;font-size:11px;font-weight:700;
                     letter-spacing:0.5px;text-transform:uppercase;">{sentiment}</span>
        <span style="background:{c_bg};color:{c_fg};padding:3px 11px;
                     border-radius:12px;font-size:11px;font-weight:700;
                     letter-spacing:0.5px;text-transform:uppercase;">{c_label}</span>
        <span style="font-size:12px;color:#9399b2;">→ {_esc(summary)}</span>
      </div>
    </div>
    """))


def memory_facts_card(facts_block: str,
                      title: str = "🧬 Director History (from Graphiti)") -> None:
    """Styled card for the Graphiti facts block — superseded items dimmed."""
    fg, bg = _PALETTE["memory"]
    if not facts_block or "no prior facts" in facts_block.lower():
        body = ('<div style="opacity:0.6;font-style:italic;color:#9399b2;">'
                'No prior facts — cold start.</div>')
    else:
        parts = []
        for raw in facts_block.split("\n"):
            line = raw.strip().lstrip("•").strip()
            if not line:
                continue
            superseded = "superseded" in line.lower()
            opacity = "0.55" if superseded else "1.0"
            badge = ('<span style="margin-left:8px;font-size:9px;padding:2px 7px;'
                     'background:#f38ba822;color:#f38ba8;border-radius:8px;'
                     'letter-spacing:0.6px;font-weight:600;">SUPERSEDED</span>'
                     if superseded else "")
            parts.append(
                f'<div style="margin:5px 0;padding:6px 12px;opacity:{opacity};'
                f'border-left:2px solid {fg};font-size:13px;color:#cdd6f4;'
                f'line-height:1.5;">'
                f'<span style="color:{fg};margin-right:8px;font-weight:700;">▸</span>'
                f'{_esc(line)}{badge}</div>'
            )
        body = "".join(parts) if parts else '<div style="opacity:0.6;">(empty)</div>'
    display(HTML(f"""
    <div style="margin:10px 0;padding:14px 18px;background:{bg};
                border-radius:10px;border:1px solid {fg}33;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:12px;color:{fg};font-weight:700;
                  letter-spacing:1.2px;text-transform:uppercase;margin-bottom:10px;">
        {title}
      </div>
      {body}
    </div>
    """))


def status_line(message: str, color_key: str = "memory", icon: str = "ℹ️") -> None:
    """Thin colored status line for agent progress messages."""
    fg, bg = _PALETTE[color_key]
    display(HTML(f"""
    <div style="margin:4px 0;padding:7px 14px;background:{bg};
                border-left:3px solid {fg};border-radius:5px;
                font-family:'Segoe UI',sans-serif;font-size:12.5px;
                color:#cdd6f4;">
      <span style="margin-right:8px;">{icon}</span>{_esc(message)}
    </div>
    """))


def direction_options_card(options: list) -> None:
    """3 creative-direction options rendered as a numbered card list."""
    fg, bg = _PALETTE["direction"]
    parts = []
    for i, opt in enumerate(options, 1):
        name = opt.get("name", "?")
        rationale = opt.get("rationale", "")
        parts.append(f"""
        <div style="margin:8px 0;padding:11px 16px;background:#11111b;
                    border-left:3px solid {fg};border-radius:6px;">
          <div style="display:flex;align-items:baseline;gap:12px;">
            <span style="background:{fg};color:#11111b;padding:2px 10px;
                         border-radius:12px;font-weight:700;font-size:12px;">{i}</span>
            <span style="color:#cdd6f4;font-weight:600;font-size:14px;">{_esc(name)}</span>
          </div>
          <div style="margin-top:8px;margin-left:36px;font-size:12.5px;
                      color:#9399b2;line-height:1.55;">{_esc(rationale)}</div>
        </div>
        """)
    display(HTML(f"""
    <div style="margin:10px 0;padding:14px 18px;background:{bg};
                border-radius:10px;border:1px solid {fg}33;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:12px;color:{fg};font-weight:700;
                  letter-spacing:1.2px;text-transform:uppercase;margin-bottom:8px;">
        🎨 Creative Direction Options
      </div>
      {"".join(parts)}
    </div>
    """))


def decision_card(decision: str, kind: str, artifact_id: str,
                  approved: bool = True) -> None:
    """Final approval / revision-requested celebratory card."""
    fg, bg = _PALETTE["approve" if approved else "revise"]
    icon = "✅" if approved else "🔄"
    label = "APPROVED" if approved else "REVISION REQUESTED"
    display(HTML(f"""
    <div style="margin:16px 0;padding:20px 24px;
                background:linear-gradient(135deg,{bg} 0%,#181825 100%);
                border:2px solid {fg};border-radius:12px;text-align:center;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:34px;">{icon}</div>
      <div style="font-size:13px;color:{fg};font-weight:700;letter-spacing:2.5px;
                  margin-top:6px;">{label}</div>
      <div style="font-size:15px;color:#cdd6f4;margin-top:10px;line-height:1.5;">{_esc(decision)}</div>
      <div style="font-size:11px;color:#9399b2;margin-top:8px;opacity:0.7;
                  letter-spacing:0.5px;">{_esc(kind)} · {_esc(artifact_id)}</div>
    </div>
    """))


def complete_banner() -> None:
    """🏁 banner that marks the end of a successful workflow run."""
    fg, bg = _PALETTE["approve"]
    display(HTML(f"""
    <div style="margin:20px 0;padding:24px;
                background:linear-gradient(135deg,{bg} 0%,#181825 100%);
                border:2px solid {fg};border-radius:14px;text-align:center;
                font-family:'Segoe UI',sans-serif;">
      <div style="font-size:36px;">🏁</div>
      <div style="font-size:20px;color:{fg};font-weight:700;margin-top:6px;
                  letter-spacing:1px;">Workflow Complete</div>
      <div style="font-size:12px;color:#9399b2;margin-top:6px;">
        All artifacts logged to Graphiti · Director history updated
      </div>
    </div>
    """))


print("🎨 Display utilities loaded — agents will now render as colored cards.")


🎨 Display utilities loaded — agents will now render as colored cards.


## 2c. 🧩 Create Hybrid Indexes (Vector + Fulltext) on Movies
For **hybrid search** we need two Neo4j indexes on `Movie` nodes:
- a **vector index** on `Movie.embedding` (semantic similarity)
- a **fulltext index** on `Movie.title` and `Movie.overview` (keyword/lexical)

`neo4j-graphrag` then fuses both signals in `HybridCypherRetriever`.

In [ ]:
MOVIE_VECTOR_INDEX   = "movie_embedding_index"
MOVIE_FULLTEXT_INDEX = "movie_text_index"

# Vector index on Movie.embedding
create_vector_index(
    driver,
    name=MOVIE_VECTOR_INDEX,
    label="Movie",
    embedding_property="embedding",
    dimensions=EMBEDDING_DIM,
    similarity_fn="cosine",
    fail_if_exists=False,
    neo4j_database=NEO4J_DATABASE,
)

# Fulltext index on Movie.title + Movie.overview
create_fulltext_index(
    driver,
    name=MOVIE_FULLTEXT_INDEX,
    label="Movie",
    node_properties=["title", "overview"],
    fail_if_exists=False,
    neo4j_database=NEO4J_DATABASE,
)
print("🧩 Hybrid indexes ensured:", MOVIE_VECTOR_INDEX, "+", MOVIE_FULLTEXT_INDEX)

🧩 Hybrid indexes ensured: movie_embedding_index + movie_text_index


## 2d. 🧮 Backfill Embeddings on Movies (one-time)
Compute embeddings for any `Movie` that doesn't have one yet. Re-runnable: only fills the missing ones. Set `MAX_TO_EMBED` to limit cost while developing.

In [ ]:
MAX_TO_EMBED = int(os.getenv("MAX_TO_EMBED", "200"))  # cap for cost control
BATCH_SIZE   = 32

def _movie_text(row: dict) -> str:
    parts = [row.get("title") or "", row.get("overview") or ""]
    genres = row.get("genres") or []
    if genres:
        parts.append("Genres: " + ", ".join([g for g in genres if g]))
    return "\n".join([p for p in parts if p]).strip()

with driver.session(database=NEO4J_DATABASE) as session:
    todo = session.run(
        """
        MATCH (m:Movie)
        WHERE m.embedding IS NULL
          AND (m.title IS NOT NULL OR m.overview IS NOT NULL)
        OPTIONAL MATCH (m)-[:HAS_GENRE]->(g:Genre)
        WITH m, collect(g.name) AS genres
        RETURN elementId(m) AS eid, m.title AS title,
               m.overview AS overview, genres
        LIMIT $limit
        """,
        limit=MAX_TO_EMBED,
    ).data()

print(f"🧮 Movies needing embeddings: {len(todo)}")

for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i : i + BATCH_SIZE]
    texts = [_movie_text(r) for r in batch]

    # Prefer batch API if the embedder exposes it; otherwise fall back per-item
    if hasattr(embedder, "embed_documents"):
        vectors = embedder.embed_documents(texts)
    else:
        vectors = [embedder.embed_query(t) for t in texts]

    upsert_vectors(
        driver,
        ids=[r["eid"] for r in batch],
        embedding_property="embedding",
        embeddings=vectors,
        entity_type=EntityType.NODE,
        neo4j_database=NEO4J_DATABASE,
    )
    print(f"  • embedded {min(i + BATCH_SIZE, len(todo))}/{len(todo)}")

print("✅ Embedding backfill complete.")

🧮 Movies needing embeddings: 0
✅ Embedding backfill complete.


## 3. 🧑‍🎨 Human Creative Director
You're the source of the wild idea and final approvals.

In [ ]:
def get_wild_idea() -> str:
    idea = input("🎬 Enter your wild movie idea: ").strip()
    if not idea:
        raise ValueError("Wild idea cannot be empty.")
    status_line(f"Wild idea captured: {idea!r}", color_key="intelligence", icon="💡")
    return idea


def get_human_feedback(stage: str, preview: str | None = None) -> str:
    """🧑‍🎨 MANDATORY free-text checkpoint before every agent acts.

    The Creative Director can:
      • approve  ("ok", "looks good", "go ahead", …)
      • dislike  ("no", "boring", "change the tone", …)
      • request specific changes  ("make it darker", "add a sidekick", …)

    Anything the human types is captured verbatim, persisted to the graph,
    and folded into the next agent's prompt. Empty input is NOT allowed.
    """
    # Pretty checkpoint banner (HTML card) replaces the dashed print line.
    checkpoint_banner(stage, preview=preview)
    while True:
        text = input(
            f"💬 Your feedback for the {stage} step "
            f"(likes / dislikes / changes — MANDATORY): "
        ).strip()
        if text:
            return text
        print("⚠️ Feedback is required — please type something (e.g. 'ok', or 'make it darker').")

## 4. 🧠 Movie Intelligence Agent — Hybrid Search **steered by Graphiti recall**

Pure `neo4j-graphrag` `HybridCypherRetriever` (vector + fulltext + `HAS_GENRE` Cypher expansion) — **no hand-rolled boost weights** anymore. Cross-session steering happens via the **query text**: before searching, we ask `graphiti.search(idea)` for the director's most relevant past facts and prepend them to the retriever query. Graphiti's bi-temporal model means stale opinions are automatically de-emphasised.

Schema this agent reads:
- `(:Movie {title, overview, rating, popularity, embedding})`, `(:Genre {name})`, `(:Movie)-[:HAS_GENRE]->(:Genre)`

Schema this agent **does not touch**: `:Entity` / `:Episodic` / `:RELATES_TO` (those are Graphiti's, only the workflow layer reads/writes them).


In [ ]:
# Vanilla hybrid retrieval Cypher — just enrich each Movie hit with its genres.
# All cross-session steering happens through the QUERY TEXT (Graphiti recall +
# director feedback are prepended), not via Cypher-side score boosts.
MOVIE_RETRIEVAL_QUERY = """
OPTIONAL MATCH (node)-[:HAS_GENRE]->(g:Genre)
WITH node, score, collect(DISTINCT g.name) AS genres
RETURN node.title       AS title,
       node.overview    AS overview,
       node.rating      AS rating,
       node.popularity  AS popularity,
       genres           AS genres,
       score            AS score
ORDER BY score DESC
"""

movie_hybrid_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=MOVIE_VECTOR_INDEX,
    fulltext_index_name=MOVIE_FULLTEXT_INDEX,
    retrieval_query=MOVIE_RETRIEVAL_QUERY,
    embedder=embedder,
    neo4j_database=NEO4J_DATABASE,
)


# --------------------------------------------------------------------------- #
# 🔧 Lucene query sanitiser                                                   #
# --------------------------------------------------------------------------- #
# Neo4j's fulltext index is backed by Lucene. These characters are reserved   #
# and crash the parser if passed raw (e.g. ":" in "WILD IDEA:" → ParseError).#
# We strip them from the keyword-side query. The rich semantic context is    #
# still preserved on the vector side via `query_vector` below.                #
import re
_LUCENE_SPECIALS = re.compile(r'[+\-&|!(){}\[\]^"~*?:\\/]|&&|\|\|')

def _lucene_safe(text: str) -> str:
    """Strip Lucene-reserved chars and collapse whitespace so the fulltext
    index receives a plain bag of keywords."""
    cleaned = _LUCENE_SPECIALS.sub(" ", text)
    # also drop bullets / em-dashes / smart punctuation that confuse Lucene
    cleaned = re.sub(r"[•—–\u2022\u2013\u2014]", " ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


async def movie_intelligence_search(idea: str, top_k: int = 8,
                                    human_feedback: str | None = None) -> pd.DataFrame:
    """🧠 Hybrid retrieval whose semantic side is steered by Graphiti recall +
    the latest director feedback, while the keyword/Lucene side gets a
    sanitised query so we never hit a ParseException."""
    # 🌐 Pull the most relevant facts Graphiti already knows about the director
    facts = await graphiti_recall(idea, num_results=5)
    facts_block = format_graphiti_facts(facts)

    # Rich context → ONLY for the vector embedder (semantic side)
    parts = [f"WILD IDEA: {idea}"]
    if facts_block and facts_block != "(no prior facts yet — cold start)":
        parts.append(f"DIRECTOR HISTORY (from Graphiti):\n{facts_block}")
    if human_feedback:
        parts.append(f"LATEST DIRECTOR FEEDBACK: {human_feedback}")
    rich_text = "\n\n".join(parts)

    # Embed the rich context ourselves → drives the vector side of the hybrid
    query_vector = embedder.embed_query(rich_text)

    # Lucene-safe keyword query → drives the fulltext side of the hybrid.
    # We mash idea + feedback + facts together as plain words, no punctuation.
    keyword_text = _lucene_safe(
        " ".join([idea, human_feedback or "", facts_block or ""])
    ) or _lucene_safe(idea) or "movie"

    result = movie_hybrid_retriever.search(
        query_text=keyword_text,
        query_vector=query_vector,
        top_k=top_k,
    )

    rows = []
    for item in result.items:
        meta = item.metadata or {}
        if not meta:
            try:
                meta = json.loads(item.content)
            except Exception:
                meta = {"content": item.content}
        rows.append(meta)

    df = pd.DataFrame(rows)
    if "genres" in df.columns:
        df_exp = df.explode("genres").rename(columns={"genres": "genre"})
    else:
        df_exp = df


    status_line(
        f"Hybrid retrieval returned {len(df)} movies → {len(df_exp)} (movie, genre) rows"
        + (f"  ·  steered by feedback: “{human_feedback}”" if human_feedback else ""),
        color_key="intelligence", icon="🧠",
    )
    return df_exp


## 5. 🎨 Creative Direction Agent — LLM-powered
Asks the LLM to synthesise **3 distinct creative directions** grounded in the hybrid-retrieved movies (titles + genres + ratings). The user then picks one.

In [ ]:
def _format_movie_context(movie_patterns: pd.DataFrame, n: int = 8) -> str:
    if movie_patterns is None or movie_patterns.empty:
        return "(no related movies retrieved)"
    cols = [c for c in ["title", "genre", "rating", "popularity", "score"]
            if c in movie_patterns.columns]
    sample = movie_patterns[cols].drop_duplicates().head(n)
    return sample.to_string(index=False)


def propose_creative_directions(idea: str, movie_patterns: pd.DataFrame,
                                human_feedback: str = "",
                                graphiti_facts_block: str = "") -> str:
    context = _format_movie_context(movie_patterns)
    facts   = graphiti_facts_block or "(no prior facts yet — cold start)"

    prompt = f"""You are the Creative Direction Agent for a film studio.

WILD IDEA from the human creative director:
{idea}

DIRECTOR HISTORY — temporal facts retrieved from Graphiti memory
(newer/non-superseded facts have priority over older ones):
{facts}

LATEST DIRECTOR FEEDBACK (RESPECT THIS):
{human_feedback or "(none yet)"}

RELATED MOVIES from the studio's Neo4j graph (hybrid retrieval):
{context}

TASK:
Propose exactly 3 distinct creative directions for this idea. Each option MUST:
  • lean into preferences shown by the DIRECTOR HISTORY facts,
  • avoid anything the history flags as disliked or superseded,
  • cite at least one history fact OR one related movie.
Return STRICT JSON only, no prose:

{{"options": [
  {{"name": "<short label>", "rationale": "<1-2 sentences>"}},
  {{"name": "...", "rationale": "..."}},
  {{"name": "...", "rationale": "..."}}
]}}"""
    raw = llm.invoke(prompt).content.strip()

    try:
        start, end = raw.find("{"), raw.rfind("}") + 1
        data = json.loads(raw[start:end])
        options = data["options"]
    except Exception:
        status_line("Could not parse LLM JSON, falling back to genre heuristic.",
                    color_key="revise", icon="⚠️")
        top = (
            movie_patterns["genre"].dropna().value_counts().head(3).index.tolist()
            if "genre" in movie_patterns.columns else []
        )
        options = [{"name": f"{g} style", "rationale": "based on dominant genre"} for g in top] \
                  or [{"name": "Drama style", "rationale": "default"}]

    # Pretty render of the 3 numbered options as a card list
    direction_options_card(options)

    choice = input(f"Enter option number [1-{len(options)}]: ").strip() or "1"
    try:
        idx = max(1, min(len(options), int(choice))) - 1
    except ValueError:
        idx = 0
    direction = options[idx]["name"]
    status_line(f"Creative direction chosen: {direction}",
                color_key="direction", icon="✅")
    return direction


## 6. ✍️ Scripting Agent — LLM-powered
Asks the LLM to draft a 3-act script outline, grounded in the wild idea, the chosen creative direction, and the top movie references retrieved by the Movie Intelligence Agent.

In [ ]:
def generate_script(creative_direction: str, idea: str,
                    movie_patterns: pd.DataFrame, human_feedback: str = "") -> str:
    context = _format_movie_context(movie_patterns, n=5)
    prompt = f"""You are the Scripting Agent.

WILD IDEA: {idea}
CREATIVE DIRECTION: {creative_direction}

DIRECTOR FEEDBACK (likes / dislikes / asks — RESPECT THIS):
{human_feedback or "(none yet)"}

REFERENCE MOVIES (hybrid retrieved from Neo4j):
{context}

TASK:
Write a tight 3-act script OUTLINE (~200 words) labelled "Act 1", "Act 2", "Act 3".
Include: protagonist, inciting incident, midpoint twist, climax, resolution.
Reference 1-2 of the related movies for tone/structure. Plain text only."""
    script = llm.invoke(prompt).content.strip()
    script = f"=== SCRIPT OUTLINE ===\nIdea: {idea}\nDirection: {creative_direction}\n\n{script}"
    artifact_card(
        title=f"Script Outline · direction: {creative_direction}",
        body=script, color_key="script", icon="✍️",
    )
    return script

## 7. 🎞️ Screenplay Agent — LLM-powered
Asks the LLM to break the script outline into 6 cinematic scenes with `INT/EXT — LOCATION — TIME` slug lines.

In [ ]:
def generate_screenplay(script: str, human_feedback: str = "") -> str:
    prompt = f"""You are the Screenplay Agent.

SCRIPT OUTLINE:
{script}

DIRECTOR FEEDBACK (likes / dislikes / asks — RESPECT THIS):
{human_feedback or "(none yet)"}

TASK:
Convert the outline into EXACTLY 6 numbered scenes. Each scene MUST start with
"Scene N — " followed by a slug line in screenplay format, e.g.:
  Scene 1 — INT. PROTAGONIST'S APARTMENT — DAY. <1-2 sentence beat>

Plain text only. No extra prose before or after the 6 scenes."""
    body = llm.invoke(prompt).content.strip()
    screenplay = "=== SCREENPLAY ===\n" + body
    artifact_card(
        title="Screenplay · 6 scenes with slug lines",
        body=screenplay, color_key="screenplay", icon="🎞️",
    )
    return screenplay

## 8. 🖼️ Storyboarding Agent — LLM-powered
Asks the LLM to turn each screenplay scene into a storyboard panel description (shot, framing, key visual, action).

In [ ]:
def create_storyboard(screenplay: str, human_feedback: str = "") -> str:
    prompt = f"""You are the Storyboarding Agent.

SCREENPLAY:
{screenplay}

DIRECTOR FEEDBACK (likes / dislikes / asks — RESPECT THIS):
{human_feedback or "(none yet)"}

TASK:
For EACH "Scene N — ..." line, produce one storyboard panel of the form:
  PANEL N — Shot: <wide/medium/close-up/POV>; Framing: <...>;
            Key Visual: <...>; Action: <one sentence>.

Output the panels in order, one per line, plain text only."""
    body = llm.invoke(prompt).content.strip()
    storyboard = "=== STORYBOARD ===\n" + body
    artifact_card(
        title="Storyboard · panels per scene",
        body=storyboard, color_key="storyboard", icon="🖼️",
    )
    return storyboard

## 9. 🌐 Memory Layer — **Graphiti only**

All session memory (artifacts, feedback, approvals, director preferences) is stored by **Graphiti** as a temporal knowledge graph **inside the same Neo4j database** as your `:Movie`/`:Genre` graph.

**What Graphiti gives us (for free, via its own Cypher under the hood):**
- `(:Entity)` / `(:Episodic)` / `(:Community)` nodes + `:RELATES_TO` / `:MENTIONS` edges.
- Each `:RELATES_TO` edge carries `name`, `fact`, `valid_at`, `invalid_at`, `created_at` — **bi-temporal** by construction.
- Vector + fulltext indexes (`entity_name_and_summary`, `episode_content`, …) created automatically by `build_indices_and_constraints()`.
- LLM-driven entity & relation extraction from any text "episode" — we never hand-write schema for "dark tone", "anti-hero", "single-location thriller", etc.
- Contradiction handling — when a newer episode contradicts an older fact, the older edge's `invalid_at` is set instead of being overwritten.

The next cell connects Graphiti to your existing Neo4j credentials and builds its indexes. That's the only setup step.


In [ ]:
import asyncio
from datetime import datetime, timezone

GRAPHITI_GROUP_ID = "movie-storyboarding"   # logical namespace inside Graphiti
DIRECTOR_NAME     = "HumanCreativeDirector" # who the facts are about

# Install once (re-running this cell is a no-op if already installed)
import importlib
if importlib.util.find_spec("graphiti_core") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "graphiti-core"])

from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType
from graphiti_core.llm_client.openai_client import OpenAIClient
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig

graphiti = Graphiti(
    uri=NEO4J_URI,
    user=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    llm_client=OpenAIClient(
        config=LLMConfig(
            api_key=OPENAI_API_KEY,
            model='gpt-5-mini',         # main model (entity / edge extraction)
            # ⚠️ Graphiti uses a SECOND, cheaper model for sub-tasks (summaries,
            # deduplication, etc). Its default is "gpt-4.1-nano", which many
            # OpenAI projects don't have access to → 403 PermissionDeniedError.
            # Pin it to a model your project CAN call.
            small_model='gpt-5-mini',
        ),
    ),
    embedder=OpenAIEmbedder(
        config=OpenAIEmbedderConfig(
            api_key=OPENAI_API_KEY,
            embedding_model=OPENAI_EMBED_MODEL,
            embedding_dim=EMBEDDING_DIM,
        ),
    ),
)

# One-time: create Graphiti's own constraints + vector/fulltext indexes inside
# the SAME Neo4j database. Safe to re-run.
await graphiti.build_indices_and_constraints()
print("🌐 Graphiti connected to Neo4j at", NEO4J_URI)
print("   → :Entity / :Episodic / :RELATES_TO live alongside :Movie in DB:", NEO4J_DATABASE)

🌐 Graphiti connected to Neo4j at bolt://34.201.26.185
   → :Entity / :Episodic / :RELATES_TO live alongside :Movie in DB: neo4j


In [ ]:
# --- Thin Graphiti wrappers ---------------------------------------------------
# Every "memory write" in the workflow becomes a single graphiti.add_episode()
# call. Graphiti's LLM extracts entities + relations and writes them into
# Neo4j with bi-temporal validity. We never hand-craft a Cypher MERGE again.
import time

async def log_artifact_episode(kind: str, content: str, artifact_id: str,
                               session_id: str, version: str = "V1",
                               supersedes_id: str | None = None,
                               referenced_titles: list[str] | None = None) -> None:
    """Log a creative artifact (CreativeDirection / Script / Screenplay / Storyboard)
    as a Graphiti episode."""
    refs = ", ".join(referenced_titles or []) or "(none)"
    sup  = f"\nThis artifact SUPERSEDES previous artifact {supersedes_id}." if supersedes_id else ""
    body = (
        f"Director: {DIRECTOR_NAME}\n"
        f"Session:  {session_id}\n"
        f"Artifact: kind={kind}, id={artifact_id}, version={version}\n"
        f"Grounded on movies: {refs}{sup}\n\n"
        f"Content:\n{content[:2500]}"
    )
    await graphiti.add_episode(
        name=f"{kind.lower()}-{artifact_id}",
        episode_body=body,
        source=EpisodeType.text,
        source_description=f"{kind} artifact ({version})",
        reference_time=datetime.now(timezone.utc),
        group_id=GRAPHITI_GROUP_ID,
    )


async def log_feedback_episode(stage: str, fb_text: str, fb_cls: dict,
                               session_id: str,
                               about_artifact_id: str | None = None,
                               referenced_titles: list[str] | None = None) -> None:
    """Log director feedback as a Graphiti episode. Graphiti's LLM will turn
    likes/dislikes/requests into preference facts on its own."""
    refs = ", ".join(referenced_titles or []) or "(none)"
    body = (
        f"Director: {DIRECTOR_NAME}\n"
        f"Session:  {session_id}\n"
        f"Stage:    {stage}\n"
        f"About artifact: {about_artifact_id or '(none)'}\n"
        f"Movies in context: {refs}\n"
        f"Sentiment (classified): {fb_cls.get('sentiment')}, "
        f"change_requested={fb_cls.get('change_requested')}\n"
        f"Summary: {fb_cls.get('summary')}\n\n"
        f"Verbatim feedback from {DIRECTOR_NAME}:\n\"\"\"{fb_text}\"\"\""
    )
    await graphiti.add_episode(
        name=f"feedback-{stage}-{uuid.uuid4().hex[:6]}",
        episode_body=body,
        source=EpisodeType.text,
        source_description=f"director feedback at {stage}",
        reference_time=datetime.now(timezone.utc),
        group_id=GRAPHITI_GROUP_ID,
    )


async def log_decision_episode(decision: str, artifact_id: str, kind: str,
                               session_id: str) -> None:
    """Log a final approval / revision decision as its own episode so it is
    discoverable by future graphiti.search() calls."""
    body = (
        f"Director: {DIRECTOR_NAME}\n"
        f"Session:  {session_id}\n"
        f"Decision: {decision}\n"
        f"On artifact: kind={kind}, id={artifact_id}"
    )
    await graphiti.add_episode(
        name=f"decision-{artifact_id}",
        episode_body=body,
        source=EpisodeType.text,
        source_description=f"decision on {kind} {artifact_id}",
        reference_time=datetime.now(timezone.utc),
        group_id=GRAPHITI_GROUP_ID,
    )


# ⚡ Session-scoped cache so the SAME (query, num_results) is only sent to
# Graphiti once. This is the biggest perf win — the workflow asks Graphiti
# about the same `wild_idea` 3-4 times across stages, and those repeat calls
# are now instant.
_GRAPHITI_RECALL_CACHE: dict[tuple[str, int], list] = {}

async def graphiti_recall(query: str, num_results: int = 5,
                          force_refresh: bool = False):
    """Hybrid (vector + BM25 + graph) search across Graphiti's temporal facts.

    Cached by (query, num_results) within this notebook session — re-asking
    the same question is essentially free. Pass `force_refresh=True` to bypass
    (e.g. after a big batch of new episodes you want to see immediately).
    """
    key = (query, num_results)
    if not force_refresh and key in _GRAPHITI_RECALL_CACHE:
        return _GRAPHITI_RECALL_CACHE[key]

    try:
        hits = await graphiti.search(
            query=query, num_results=num_results, group_ids=[GRAPHITI_GROUP_ID],
        )
    except Exception as e:
        status_line(f"Graphiti search failed: {e}", color_key="revise", icon="⚠️")
        hits = []
    _GRAPHITI_RECALL_CACHE[key] = hits
    return hits


def clear_graphiti_recall_cache() -> None:
    """Drop the session-scoped recall cache. Call this after a big batch of
    new episodes if you need the next search to see them immediately
    (Graphiti also needs a moment to extract entities from new episodes)."""
    _GRAPHITI_RECALL_CACHE.clear()


def format_graphiti_facts(hits, n: int = 5) -> str:
    """Render Graphiti search hits (EntityEdges) as a compact bullet list for prompts."""
    if not hits:
        return "(no prior facts yet — cold start)"
    lines = []
    for h in hits[:n]:
        fact     = getattr(h, "fact", "") or getattr(h, "name", "?")
        valid_at = getattr(h, "valid_at", None)
        invalid  = getattr(h, "invalid_at", None)
        marker   = " (superseded)" if invalid else ""
        when     = f" [valid_at={valid_at}]" if valid_at else ""
        lines.append(f"• {fact}{when}{marker}")
    return "\n".join(lines)


# --- LLM-based sentiment classifier for the free-text feedback ----------------
# Still useful as a quick local signal so the workflow can decide whether to
# auto-revise. Graphiti will do its own deeper extraction in parallel.
def classify_feedback(text: str) -> dict:
    """Return {'sentiment': 'positive'|'negative'|'neutral',
                'change_requested': bool,
                'summary': '...'} for the human's free-text feedback."""
    prompt = f"""Classify the following human feedback for a creative film workflow.

FEEDBACK: \"\"\"{text}\"\"\"

Return STRICT JSON only:
{{"sentiment": "positive" | "negative" | "neutral",
  "change_requested": true | false,
  "summary": "<one short sentence capturing the ask>"}}"""
    try:
        raw = llm.invoke(prompt).content.strip()
        start, end = raw.find("{"), raw.rfind("}") + 1
        data = json.loads(raw[start:end])
        return {
            "sentiment": str(data.get("sentiment", "neutral")).lower(),
            "change_requested": bool(data.get("change_requested", False)),
            "summary": data.get("summary", text[:120]),
        }
    except Exception:
        lower = text.lower()
        negative = any(w in lower for w in
                       ["no", "don't", "dont", "dislike", "boring", "change", "redo",
                        "instead", "but", "however", "darker", "lighter"])
        positive = any(w in lower for w in
                       ["ok", "okay", "good", "great", "love", "yes", "approve", "perfect"])
        if negative and not positive:
            sent, chg = "negative", True
        elif positive and not negative:
            sent, chg = "positive", False
        else:
            sent, chg = "neutral", "change" in lower or "instead" in lower
        return {"sentiment": sent, "change_requested": chg, "summary": text[:120]}


## 11. 🧬 Living Agent — recall via Graphiti

Before each agent acts, we ask Graphiti's hybrid search for the most relevant past **temporal facts** about the director (preferences, prior decisions, superseded opinions). Those facts are folded into the agent's prompt, so every agent is **stateful across sessions** without us writing a single MERGE.


In [ ]:
# async def recall_director_facts(query: str, num_results: int = 3) -> str:
#     """Pull the most relevant temporal facts Graphiti knows about the director,
#     formatted for prompt injection."""
#     hits = await graphiti_recall(query, num_results=num_results)
#     block = format_graphiti_facts(hits, n=num_results)
#     print(f" Graphiti recalled {len(hits)} past fact(s) for query: {query!r}")
#     return block

async def recall_director_facts(query: str, num_results: int = 3,
                                force_refresh: bool = False) -> str:
    """Pull the most relevant temporal facts Graphiti knows about the director,
    formatted for prompt injection. Cached by (query, num_results) within this
    notebook session — re-asking the same question is essentially free.
    """
    hits = await graphiti_recall(query, num_results=num_results,
                                 force_refresh=force_refresh)
    block = format_graphiti_facts(hits, n=num_results)
    return block


## 12. 🔄 Versioning — handled by Graphiti

When the director rejects an artifact and we regenerate it, we simply log a **new** episode that says *"this artifact supersedes the previous one"*. Graphiti's LLM extracts that as a temporal fact, the older fact's `invalid_at` is set automatically, and `graphiti.search()` will surface the newest valid one. **No `:SUPERSEDES` edges, no manual versioning code.**


In [ ]:
# Versioning is just another episode — Graphiti handles invalid_at automatically.
async def log_revision_episode(old_artifact_id: str, new_artifact_id: str,
                               kind: str, new_content: str, new_version: str,
                               session_id: str, reason: str,
                               referenced_titles: list[str] | None = None) -> None:
    """Log that new_artifact_id supersedes old_artifact_id, plus the new content.
    Graphiti's LLM will extract a 'supersedes' relation with valid_at = now and
    set invalid_at on any contradicting older fact."""
    refs = ", ".join(referenced_titles or []) or "(none)"
    body = (
        f"Director: {DIRECTOR_NAME}\n"
        f"Session:  {session_id}\n"
        f"REVISION: artifact {new_artifact_id} (version {new_version}, kind {kind}) "
        f"supersedes the previous artifact {old_artifact_id}.\n"
        f"Reason for revision: {reason}\n"
        f"Movies in context: {refs}\n\n"
        f"New content:\n{new_content[:2500]}"
    )
    await graphiti.add_episode(
        name=f"revision-{new_artifact_id}",
        episode_body=body,
        source=EpisodeType.text,
        source_description=f"revision: {kind} {old_artifact_id} → {new_artifact_id}",
        reference_time=datetime.now(timezone.utc),
        group_id=GRAPHITI_GROUP_ID,
    )


## 13. 🚦 Main Agentic Workflow — Human-in-the-Loop, Graphiti-Backed

**Mandatory human checkpoint before every agent.** The Creative Director's free-text feedback is:
1. captured via `get_human_feedback(...)` (empty input is rejected),
2. classified locally as `positive` / `negative` / `neutral` (with `change_requested` flag) so the workflow can decide whether to auto-revise,
3. **logged as a Graphiti episode** — Graphiti's LLM extracts director preferences as `:Entity` + `:RELATES_TO` facts with bi-temporal validity,
4. **folded into the next agent's prompt** so the agent acts on the feedback,
5. if the feedback signals a dislike or change request, the previous artifact is re-generated and a **revision episode** is logged. Graphiti's `invalid_at` mechanism handles versioning automatically.

At the **start** of every session, `graphiti.search(wild_idea)` recalls the director's most relevant past facts so the workflow is **stateful across sessions** with zero hand-rolled schema.


In [ ]:
session_id = f"session-{uuid.uuid4().hex[:8]}"
session_banner(session_id)

# Fresh session = fresh recall cache (so we see any episodes from previous runs)
clear_graphiti_recall_cache()

# Movies the Intelligence Agent surfaced; surfaced into every artifact episode
referenced_titles: list[str] = []


# --------------------------------------------------------------------------- #
# Helper: human checkpoint → classify → log a Graphiti feedback episode       #
# --------------------------------------------------------------------------- #
async def human_checkpoint(stage: str,
                           about_artifact_id: str | None = None,
                           preview: str | None = None) -> tuple[str, dict]:
    text = get_human_feedback(stage, preview=preview)
    cls  = classify_feedback(text)
    feedback_pill(text, cls["sentiment"], cls["change_requested"], cls["summary"])
    await log_feedback_episode(
        stage=stage, fb_text=text, fb_cls=cls,
        session_id=session_id,
        about_artifact_id=about_artifact_id,
        referenced_titles=referenced_titles,
    )
    status_line(
        f"Feedback logged to Graphiti"
        + (f" (about {about_artifact_id})" if about_artifact_id else ""),
        color_key="memory", icon="🌐",
    )
    return text, cls


# --------------------------------------------------------------------------- #
# Helper: regenerate an artifact + log a revision episode                     #
# --------------------------------------------------------------------------- #
async def revise_artifact(old_id: str, kind: str, regen_callable,
                          reason: str, next_version: str = "V2") -> tuple[str, str]:
    new_content = regen_callable()
    new_id      = f"{kind.lower()}-{uuid.uuid4().hex[:6]}"
    await log_revision_episode(
        old_artifact_id=old_id, new_artifact_id=new_id,
        kind=kind, new_content=new_content, new_version=next_version,
        session_id=session_id, reason=reason,
        referenced_titles=referenced_titles,
    )
    status_line(f"{new_id} ({next_version}) supersedes {old_id}  ·  reason: {reason}",
                color_key="revise", icon="🔄")
    return new_id, new_content


# =========================================================================== #
# STAGE 0 — Human Creative Director provides the WILD IDEA                    #
# =========================================================================== #
stage_banner(0, "Wild Idea + Memory Recall", color_key="memory",
             subtitle="Capture the seed concept and recall the director's past preferences")
wild_idea = get_wild_idea()

# 🧬 Recall everything Graphiti already knows that's relevant to this idea.
director_facts_block = await recall_director_facts(wild_idea, num_results=6)
memory_facts_card(director_facts_block)

# Checkpoint BEFORE Movie Intelligence Agent
fb_text, fb_cls = await human_checkpoint(
    "before-MovieIntelligence",
    preview=f"Wild idea: {wild_idea}\n\nGraphiti facts:\n{director_facts_block}",
)

# =========================================================================== #
# STAGE 1 — 🧠 Movie Intelligence Agent (Graphiti-steered hybrid)             #
# =========================================================================== #
stage_banner(1, "🧠 Movie Intelligence Agent", color_key="intelligence",
             subtitle="Hybrid vector + fulltext retrieval over the Neo4j movie graph")
movie_patterns = await movie_intelligence_search(wild_idea, top_k=8, human_feedback=fb_text)
display(movie_patterns)
if not movie_patterns.empty and "title" in movie_patterns.columns:
    referenced_titles = movie_patterns["title"].dropna().unique().tolist()[:10]

# Checkpoint AFTER Intelligence / BEFORE Creative Direction
fb_text, fb_cls = await human_checkpoint(
    "before-CreativeDirection",
    preview=(f"Top retrieved:\n{movie_patterns.head(5).to_string(index=False)}"
             if not movie_patterns.empty else "(none)"),
)
if fb_cls["change_requested"] or fb_cls["sentiment"] == "negative":
    status_line("Re-running hybrid retrieval with director feedback…",
                color_key="intelligence", icon="🔁")
    movie_patterns = await movie_intelligence_search(wild_idea, top_k=8, human_feedback=fb_text)
    display(movie_patterns)
    if not movie_patterns.empty and "title" in movie_patterns.columns:
        referenced_titles = movie_patterns["title"].dropna().unique().tolist()[:10]

# =========================================================================== #
# STAGE 2 — 🎨 Creative Direction Agent                                       #
# =========================================================================== #
stage_banner(2, "🎨 Creative Direction Agent", color_key="direction",
             subtitle="LLM proposes 3 directions, you pick one")
creative_direction = propose_creative_directions(
    wild_idea, movie_patterns, human_feedback=fb_text,
    graphiti_facts_block=director_facts_block,
)
cd_id = f"cd-{uuid.uuid4().hex[:6]}"
await log_artifact_episode(
    kind="CreativeDirection", content=creative_direction,
    artifact_id=cd_id, session_id=session_id, version="V1",
    referenced_titles=referenced_titles,
)

fb_text, fb_cls = await human_checkpoint(
    "before-Scripting", about_artifact_id=cd_id,
    preview=f"Creative direction: {creative_direction}",
)
if fb_cls["change_requested"] or fb_cls["sentiment"] == "negative":
    cd_id, creative_direction = await revise_artifact(
        cd_id, "CreativeDirection",
        lambda: propose_creative_directions(
            wild_idea, movie_patterns, human_feedback=fb_text,
            graphiti_facts_block=director_facts_block,
        ),
        reason=fb_cls["summary"],
    )

# =========================================================================== #
# STAGE 3 — ✍️ Scripting Agent                                                #
# =========================================================================== #
stage_banner(3, "✍️ Scripting Agent", color_key="script",
             subtitle="LLM drafts a 3-act outline grounded in movies + director taste")
script    = generate_script(creative_direction, wild_idea, movie_patterns, human_feedback=fb_text)
script_id = f"script-{uuid.uuid4().hex[:6]}"
await log_artifact_episode(
    kind="Script", content=script, artifact_id=script_id,
    session_id=session_id, version="V1", referenced_titles=referenced_titles,
)

fb_text, fb_cls = await human_checkpoint(
    "before-Screenplay", about_artifact_id=script_id, preview=script,
)
if fb_cls["change_requested"] or fb_cls["sentiment"] == "negative":
    script_id, script = await revise_artifact(
        script_id, "Script",
        lambda: generate_script(creative_direction, wild_idea, movie_patterns, human_feedback=fb_text),
        reason=fb_cls["summary"],
    )

# =========================================================================== #
# STAGE 4 — 🎞️ Screenplay Agent                                               #
# =========================================================================== #
stage_banner(4, "🎞️ Screenplay Agent", color_key="screenplay",
             subtitle="LLM converts the outline into 6 cinematic scenes")
screenplay = generate_screenplay(script, human_feedback=fb_text)
sp_id      = f"screenplay-{uuid.uuid4().hex[:6]}"
await log_artifact_episode(
    kind="Screenplay", content=screenplay, artifact_id=sp_id,
    session_id=session_id, version="V1", referenced_titles=referenced_titles,
)

fb_text, fb_cls = await human_checkpoint(
    "before-Storyboard", about_artifact_id=sp_id, preview=screenplay,
)
if fb_cls["change_requested"] or fb_cls["sentiment"] == "negative":
    sp_id, screenplay = await revise_artifact(
        sp_id, "Screenplay",
        lambda: generate_screenplay(script, human_feedback=fb_text),
        reason=fb_cls["summary"],
    )

# =========================================================================== #
# STAGE 5 — 🖼️ Storyboarding Agent                                            #
# =========================================================================== #
stage_banner(5, "🖼️ Storyboarding Agent", color_key="storyboard",
             subtitle="LLM turns every scene into a storyboard panel description")
storyboard = create_storyboard(screenplay, human_feedback=fb_text)
sb_id      = f"storyboard-{uuid.uuid4().hex[:6]}"
await log_artifact_episode(
    kind="Storyboard", content=storyboard, artifact_id=sb_id,
    session_id=session_id, version="V1", referenced_titles=referenced_titles,
)

# Final checkpoint — approve or revise
fb_text, fb_cls = await human_checkpoint(
    "final-approval", about_artifact_id=sb_id, preview=storyboard,
)
if fb_cls["sentiment"] == "positive" and not fb_cls["change_requested"]:
    await log_decision_episode(
        decision=f"APPROVED: {fb_cls['summary']}",
        artifact_id=sb_id, kind="Storyboard", session_id=session_id,
    )
    decision_card(decision=fb_cls["summary"], kind="Storyboard",
                  artifact_id=sb_id, approved=True)
else:
    await log_decision_episode(
        decision=f"REVISION REQUESTED: {fb_cls['summary']}",
        artifact_id=sb_id, kind="Storyboard", session_id=session_id,
    )
    decision_card(decision=fb_cls["summary"], kind="Storyboard",
                  artifact_id=sb_id, approved=False)
    sb_id, storyboard = await revise_artifact(
        sb_id, "Storyboard",
        lambda: create_storyboard(screenplay, human_feedback=fb_text),
        reason=fb_cls["summary"],
    )

# =========================================================================== #
# 🧬 Director history AFTER this session — shows how Graphiti's view evolved   #
# =========================================================================== #
final_facts_block = await recall_director_facts(wild_idea, num_results=8,
                                                force_refresh=True)
memory_facts_card(final_facts_block,
                  title="🧬 Director History AFTER this Session (Graphiti)")

complete_banner()


## 14. 🔎 Explore the Graphiti Memory

These queries make Graphiti's temporal memory **explainable** — you can answer "What episodes did we log this session?", "What facts has the LLM extracted about the director?", "Which facts have been superseded?", "Which Graphiti entities overlap with our `:Movie` titles?".


In [ ]:
# 🌐 Episodes Graphiti recorded for THIS session
with driver.session(database=NEO4J_DATABASE) as session:
    episodes = session.run(
        """
        MATCH (ep:Episodic)
        WHERE ep.group_id = $gid AND ep.content CONTAINS $sid
        RETURN ep.name        AS name,
               ep.source      AS source,
               ep.source_description AS description,
               ep.valid_at    AS valid_at,
               ep.created_at  AS created_at
        ORDER BY ep.created_at
        """,
        gid=GRAPHITI_GROUP_ID, sid=session_id,
    ).to_df()
episodes


In [ ]:
# 🌐 Top entities Graphiti has extracted across all sessions in this group
with driver.session(database=NEO4J_DATABASE) as session:
    top_entities = session.run(
        """
        MATCH (e:Entity)
        WHERE e.group_id = $gid OR $gid IN coalesce(e.group_ids, [])
        OPTIONAL MATCH (e)-[r:RELATES_TO]-()
        RETURN e.name    AS entity,
               e.summary AS summary,
               count(r)  AS facts
        ORDER BY facts DESC LIMIT 20
        """,
        gid=GRAPHITI_GROUP_ID,
    ).to_df()
top_entities


In [ ]:
# 🌐 Bi-temporal facts — both currently valid and superseded
with driver.session(database=NEO4J_DATABASE) as session:
    facts = session.run(
        """
        MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
        WHERE r.group_id = $gid OR $gid IN coalesce(r.group_ids, [])
        RETURN a.name      AS source,
               r.name      AS relation,
               b.name      AS target,
               r.fact      AS fact,
               r.valid_at  AS valid_at,
               r.invalid_at AS invalid_at,
               CASE WHEN r.invalid_at IS NULL THEN '✅ valid' ELSE '⏳ superseded' END AS status
        ORDER BY r.created_at DESC LIMIT 25
        """,
        gid=GRAPHITI_GROUP_ID,
    ).to_df()
facts


In [ ]:
# 🌐 Demo Graphiti's hybrid search — ask any natural-language question
demo_query = "What does the director like and dislike?"
hits = await graphiti_recall(demo_query, num_results=8)
print(f"🌐 Graphiti search for: {demo_query!r}\n")
for h in hits:
    fact     = getattr(h, "fact", None) or getattr(h, "name", "?")
    valid_at = getattr(h, "valid_at", None)
    invalid  = getattr(h, "invalid_at", None)
    marker   = "  ⏳ superseded" if invalid else "  ✅ valid"
    print(f"• {fact}{marker}  [valid_at={valid_at}]")


In [ ]:
# 🔗 CROSS-GRAPH JOIN — Movies that Graphiti has also recognised as Entities
# This is the payoff of running both subgraphs in the SAME Neo4j database.
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run(
        """
        MATCH (e:Entity) WITH count(e) AS entities
        MATCH (ep:Episodic) WITH entities, count(ep) AS episodes
        MATCH ()-[r:RELATES_TO]->() RETURN entities, episodes, count(r) AS facts
        """
    ).to_df()
    print("🌐 Graphiti subgraph counts (same DB as :Movie):")
    display(counts)

    bridge = session.run(
        """
        MATCH (m:Movie), (e:Entity)
        WHERE toLower(e.name) CONTAINS toLower(m.title)
           OR toLower(m.title) CONTAINS toLower(e.name)
        OPTIONAL MATCH (e)-[r:RELATES_TO]-()
        RETURN m.title       AS movie_title,
               e.name        AS graphiti_entity,
               e.summary     AS summary,
               count(r)      AS facts
        ORDER BY facts DESC LIMIT 15
        """
    ).to_df()
    print("\n🔗 Movies the LLM also extracted as Graphiti Entities:")
    display(bridge if not bridge.empty else "(no overlap yet — try more sessions)")


## 15. 🔒 Close the Driver

In [ ]:
try:
    await graphiti.close()
    print("👋 Graphiti closed.")
except Exception as e:
    print(f"⚠️ Graphiti close failed: {e}")

driver.close()
print("👋 Neo4j driver closed.")


## ✅ Summary
- **Two subgraphs, one Neo4j database:**
  - `(:Movie)-[:HAS_GENRE]->(:Genre)` — your existing studio graph (untouched).
  - `(:Entity)`, `(:Episodic)`, `(:RELATES_TO)` — Graphiti's auto-extracted temporal knowledge graph, written by `graphiti.add_episode(...)`.
- **Hand-rolled memory layer is gone.** No more `(:Director)`, `(:Artifact)`, `(:Feedback)`, `(:MemoryRecord)`, `[:PREFERS]`, `[:AVOIDS]`, `[:REFERENCES]`, `[:SUPERSEDES]`, `[:DROVE_REVISION]`. Every memory write in the workflow is a single `await graphiti.add_episode(...)` call.
- **Human-in-the-loop at every stage** — free-text feedback is MANDATORY before each agent. It's classified locally (positive/negative/neutral, change_requested) so the workflow can decide whether to auto-revise, then logged as a Graphiti episode.
- **Versioning is bi-temporal, not edge-based.** When the director rejects an artifact, we log a *revision episode* — Graphiti's LLM extracts a "supersedes" fact and sets `invalid_at` on any contradicting older fact automatically.
- **Cross-session living loop.** At every stage we call `graphiti.search(query)` to recall the most relevant temporal facts about the director and fold them into the next agent's prompt. The Movie Intelligence Agent prepends the same recall to its hybrid-retrieval query text.
- **Movie Intelligence Agent** uses vanilla `neo4j-graphrag` `HybridCypherRetriever` (vector + fulltext + `HAS_GENRE`). All cross-session steering happens through query text, not Cypher score boosts.
- **Cross-graph joins** are still possible — see Section 14 — because both subgraphs live in the same `NEO4J_DATABASE`.
- All creative agents are **LLM-powered** via `OpenAILLM`, grounded on hybrid-retrieved movies + human feedback + Graphiti recall.
